# Cerebrum Plugin Handover Demo

This notebook demonstrates how different AI agent frameworks can collaborate through Cerebrum's plugin system. We'll show how to:

1. Load and initialize multiple agent framework plugins
2. Create agents with different frameworks
3. Pass data between plugins in a workflow
4. Visualize the execution flow

The demo will use three different frameworks:
- **CrewAI** for initial research tasks
- **AutoGen** for data analysis
- **SmolAgents** for report generation

## Setup and Imports

In [ ]:
# Import necessary components
import sys
import os
import asyncio
import json
from typing import Dict, Any, List
import matplotlib.pyplot as plt
import networkx as nx

# Add project root to path for imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import Cerebrum components
from src.server.plugins.plugin_manager import PluginManager
from src.server.plugins.agent_framework_config import AgentFrameworkConfig

## Initialize Plugin Manager

First, we'll initialize the plugin manager, which handles the loading and management of different agent framework plugins.

In [ ]:
# Initialize plugin manager
plugin_manager = PluginManager()
print("✅ Plugin manager initialized")

## Load Agent Framework Plugins

Next, we'll load the three different agent framework plugins we'll be using in this demo.

In [ ]:
async def load_demo_plugins():
    """Load the necessary plugins for demonstration."""
    # Configure and load CrewAI
    crewai_config = AgentFrameworkConfig(
        name="crewai",
        version="1.0.0",
        description="CrewAI Plugin for multi-agent collaboration",
        settings={
            "model": "gpt-4o-mini",
            "temperature": 0.7,
            "verbose": True
        }
    )
    crewai_path = os.path.join(project_root, "src/server/plugins/crewai_plugin.py")
    crewai_success = await plugin_manager.load_plugin(crewai_path, crewai_config)
    print(f"CrewAI plugin loaded: {'✅' if crewai_success else '❌'}")
    
    # Configure and load AutoGen
    autogen_config = AgentFrameworkConfig(
        name="autogen",
        version="1.0.0",
        description="AutoGen Plugin for conversational agents",
        settings={
            "model": "gpt-4o-mini",
            "temperature": 0.5,
            "verbose": True
        }
    )
    autogen_path = os.path.join(project_root, "src/server/plugins/autogen_plugin.py")
    autogen_success = await plugin_manager.load_plugin(autogen_path, autogen_config)
    print(f"AutoGen plugin loaded: {'✅' if autogen_success else '❌'}")
    
    # Configure and load SmolAgents
    smolagents_config = AgentFrameworkConfig(
        name="smolagents",
        version="1.0.0",
        description="SmolAgents Plugin for lightweight agents",
        settings={
            "model": "gpt-4o-mini",
            "temperature": 0.6,
            "verbose": True
        }
    )
    smolagents_path = os.path.join(project_root, "src/server/plugins/smolagents_plugin.py")
    smolagents_success = await plugin_manager.load_plugin(smolagents_path, smolagents_config)
    print(f"SmolAgents plugin loaded: {'✅' if smolagents_success else '❌'}")
    
    # Return overall success status
    return crewai_success and autogen_success and smolagents_success

# Execute the async function to load plugins
success = await load_demo_plugins()
if success:
    print("\n✅ All plugins loaded successfully")
else:
    print("\n⚠️ Some plugins failed to load")

## List Available Plugins

Let's check what plugins are available and their capabilities.

In [ ]:
# List all loaded plugins
plugins = plugin_manager.list_plugins()
print(f"Loaded {len(plugins)} plugins:")

for plugin in plugins:
    print(f"\n📦 {plugin['name']} v{plugin['version']}")
    print(f"   Description: {plugin['description']}")
    print(f"   Status: {'Active' if plugin['is_initialized'] else 'Inactive'}")
    print(f"   Capabilities:")
    for capability in plugin['capabilities']:
        print(f"    - {capability}")

## Visualize Plugin Handover Flow

Now let's create a visual representation of how data will flow between the different agent frameworks in our demo.

In [ ]:
def visualize_plugin_flow():
    """Create a visual representation of plugin handover workflow."""
    # Create directed graph
    G = nx.DiGraph()
    
    # Add nodes for each plugin/task
    G.add_node("CrewAI\nResearcher", color='lightblue', plugin='crewai')
    G.add_node("AutoGen\nAnalyst", color='lightgreen', plugin='autogen')
    G.add_node("SmolAgents\nWriter", color='lightsalmon', plugin='smolagents')
    
    # Add edges showing data flow
    G.add_edge("CrewAI\nResearcher", "AutoGen\nAnalyst", 
              label="Research Results\n(JSON)")
    G.add_edge("AutoGen\nAnalyst", "SmolAgents\nWriter", 
              label="Analysis Results\n(JSON)")
    
    # Create the plot with a larger figure
    plt.figure(figsize=(12, 8))
    
    # Create a custom layout
    pos = {
        "CrewAI\nResearcher": (0, 0),
        "AutoGen\nAnalyst": (1, 0),
        "SmolAgents\nWriter": (2, 0)
    }
    
    # Draw nodes with custom colors
    node_colors = [G.nodes[n]['color'] for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=4000, alpha=0.8)
    
    # Draw edges with arrows
    nx.draw_networkx_edges(G, pos, width=2, arrowsize=25, arrowstyle='->', connectionstyle='arc3,rad=0.1')
    
    # Add node labels
    nx.draw_networkx_labels(G, pos, font_size=14, font_weight='bold')
    
    # Add edge labels
    edge_labels = {(u, v): d['label'] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=12, label_pos=0.5)
    
    # Add a title and adjust layout
    plt.title("Plugin Handover Flow", fontsize=20, pad=20)
    plt.axis('off')
    plt.tight_layout()
    
    # Add legend for plugin frameworks
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='lightblue', label='CrewAI'),
        Patch(facecolor='lightgreen', label='AutoGen'),
        Patch(facecolor='lightsalmon', label='SmolAgents')
    ]
    plt.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
    
    plt.show()

# Display the visualization
visualize_plugin_flow()

## Implement Multi-Plugin Workflow

Now we'll implement a workflow that demonstrates handover between the three different agent frameworks. The workflow will:

1. Use CrewAI to perform initial research on a topic
2. Pass the research results to AutoGen for analysis
3. Pass both the research and analysis to SmolAgents to generate a final report

In [ ]:
async def run_multi_plugin_workflow(topic="AI agents"):
    """Run a workflow that hands off between multiple plugins."""
    # Create tracking structure for results and steps
    results = {
        "steps": [],
        "outputs": {}
    }
    
    try:
        # Step 1: CrewAI for research
        print("\n🔍 Step 1: CrewAI Research")
        results["steps"].append("Creating CrewAI researcher agent")
        
        researcher_config = {
            "role": "Researcher",
            "goal": f"Find detailed information about {topic}",
            "backstory": "Expert researcher with broad knowledge of AI and technology",
            "tools": [
                {"name": "web_search", "description": "Search the web for information"}
            ]
        }
        
        researcher = await plugin_manager.create_agent("crewai", researcher_config)
        
        research_task = {
            "description": f"Research the latest developments in {topic}, focusing on real-world applications, key technologies, and future trends.",
            "expected_output": "A comprehensive research report with key findings and insights"
        }
        
        results["steps"].append(f"Executing research task with CrewAI: {research_task['description']}")
        research_result = await plugin_manager.execute_task("crewai", researcher, research_task)
        print(f"✅ CrewAI Research completed")
        
        # Extract result content or use full result object
        if isinstance(research_result, dict) and 'result' in research_result:
            research_output = research_result['result']
        else:
            research_output = research_result
            
        results["outputs"]["research"] = research_output
        print(f"📝 Research summary: {str(research_output)[:150]}...")
        
        # Step 2: AutoGen for analysis
        print("\n📊 Step 2: AutoGen Analysis")
        results["steps"].append("Creating AutoGen analyst agent")
        
        analyst_config = {
            "role": "Analyst",
            "goal": "Analyze research findings and identify key trends and patterns",
            "previous_results": results["outputs"]["research"]
        }
        
        analyst = await plugin_manager.create_agent("autogen", analyst_config)
        
        analysis_task = {
            "description": "Analyze the research findings, identify key trends, and extract actionable insights",
            "input_data": {"research": results["outputs"]["research"]}
        }
        
        results["steps"].append(f"Executing analysis task with AutoGen: {analysis_task['description']}")
        analysis_result = await plugin_manager.execute_task("autogen", analyst, analysis_task)
        print(f"✅ AutoGen Analysis completed")
        
        # Extract result content or use full result object
        if isinstance(analysis_result, dict) and 'result' in analysis_result:
            analysis_output = analysis_result['result']
        else:
            analysis_output = analysis_result
            
        results["outputs"]["analysis"] = analysis_output
        print(f"📝 Analysis summary: {str(analysis_output)[:150]}...")
        
        # Step 3: SmolAgents for report generation
        print("\n📝 Step 3: SmolAgents Report Generation")
        results["steps"].append("Creating SmolAgents writer agent")
        
        report_writer_config = {
            "role": "Writer",
            "goal": "Create a final report based on the research and analysis",
            "previous_results": {
                "research": results["outputs"]["research"],
                "analysis": results["outputs"]["analysis"]
            }
        }
        
        report_writer = await plugin_manager.create_agent("smolagents", report_writer_config)
        
        report_task = {
            "description": "Create a comprehensive final report based on the research and analysis, including executive summary, key findings, and recommendations",
            "input_data": {
                "analysis": results["outputs"]["analysis"],
                "research": results["outputs"]["research"]
            }
        }
        
        results["steps"].append(f"Executing report task with SmolAgents: {report_task['description']}")
        report_result = await plugin_manager.execute_task("smolagents", report_writer, report_task)
        print(f"✅ SmolAgents Report Generation completed")
        
        # Extract result content or use full result object
        if isinstance(report_result, dict) and 'result' in report_result:
            report_output = report_result['result']
        else:
            report_output = report_result
            
        results["outputs"]["report"] = report_output
        print(f"📝 Report summary: {str(report_output)[:150]}...")
        
        return results
    
    except Exception as e:
        print(f"❌ Error in workflow: {str(e)}")
        # Add error to results
        results["error"] = str(e)
        return results

## Execute the Multi-Plugin Workflow

Let's run the workflow with a specific topic to demonstrate the handover between plugins.

In [ ]:
# Run the workflow with a specific topic
workflow_results = await run_multi_plugin_workflow("agentic frameworks and their business applications")

## Display the Final Results

Let's examine the final report generated by our multi-plugin workflow.

In [ ]:
# Display the final report
if "report" in workflow_results["outputs"]:
    print("\n🎉 Final Report:")
    print("=" * 80)
    print(workflow_results["outputs"]["report"])
    print("=" * 80)
else:
    print("❌ No report generated in the workflow")

## Visualize Result Flow

Now let's visualize how the data actually flowed through our workflow.

In [ ]:
def visualize_result_flow(results):
    """Visualize the data flow between plugins in the executed workflow."""
    # Create figure and subplots
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Define positions for the three stages
    positions = {
        "research": (0, 0),
        "analysis": (5, 0),
        "report": (10, 0)
    }
    
    # Define colors for each framework
    colors = {
        "research": "lightblue",
        "analysis": "lightgreen",
        "report": "lightsalmon"
    }
    
    # Define labels for each stage
    labels = {
        "research": "CrewAI Research",
        "analysis": "AutoGen Analysis",
        "report": "SmolAgents Report"
    }
    
    # Draw boxes for each stage
    for stage, pos in positions.items():
        x, y = pos
        width, height = 4, 3
        
        rect = plt.Rectangle((x - width/2, y - height/2), width, height, 
                            facecolor=colors[stage], edgecolor="black", alpha=0.7,
                            linewidth=2, zorder=1)
        ax.add_patch(rect)
        
        # Add stage label
        ax.text(x, y + 0.75, labels[stage], ha="center", fontsize=14, fontweight="bold")
        
        # Add content summary (truncated)
        if stage in results["outputs"]:
            content = str(results["outputs"][stage])
            # Truncate to about 200 chars and format for display
            content = content[:200] + "..." if len(content) > 200 else content
            content = "\n".join([content[i:i+30] for i in range(0, len(content), 30)])
            ax.text(x, y - 0.5, content, ha="center", fontsize=8, wrap=True)
    
    # Draw arrows for data flow
    arrow_props = dict(arrowstyle="->", linewidth=2, color="black")
    
    # Arrow from research to analysis
    ax.annotate("", xy=(positions["analysis"][0] - 2, positions["analysis"][1]), 
               xytext=(positions["research"][0] + 2, positions["research"][1]),
               arrowprops=arrow_props)
    
    # Arrow from analysis to report
    ax.annotate("", xy=(positions["report"][0] - 2, positions["report"][1]), 
               xytext=(positions["analysis"][0] + 2, positions["analysis"][1]),
               arrowprops=arrow_props)
    
    # Add flow labels
    ax.text(2.5, 0.5, "Research Data", ha="center", fontsize=10)
    ax.text(7.5, 0.5, "Analysis Results", ha="center", fontsize=10)
    
    # Add legend for framework colors
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="lightblue", label="CrewAI"),
        Patch(facecolor="lightgreen", label="AutoGen"),
        Patch(facecolor="lightsalmon", label="SmolAgents")
    ]
    ax.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, -0.15), ncol=3)
    
    # Set axis limits and remove ticks
    ax.set_xlim(-3, 13)
    ax.set_ylim(-2, 3)
    ax.set_axis_off()
    
    # Add title
    plt.title("Multi-Plugin Workflow Results", fontsize=18, pad=20)
    
    plt.tight_layout()
    plt.show()

# Visualize the workflow results
visualize_result_flow(workflow_results)

## Analyzing Plugin Handover Performance

Let's analyze the workflow to understand how effectively the plugins handed off data to each other.

In [ ]:
def analyze_workflow_performance(results):
    """Analyze the workflow performance and data transfer between plugins."""
    print("📊 Workflow Performance Analysis")
    print("=" * 80)
    
    # Check if all steps were completed
    expected_outputs = ["research", "analysis", "report"]
    completed_outputs = [out for out in expected_outputs if out in results["outputs"]]
    
    print(f"Steps completed: {len(completed_outputs)}/{len(expected_outputs)}")
    
    # Analyze output sizes
    print("\nOutput sizes:")
    for output_key in completed_outputs:
        output_content = results["outputs"][output_key]
        output_size = len(str(output_content))
        print(f"  {output_key.capitalize()}: {output_size} characters")
    
    # Analyze content richness
    print("\nContent analysis:")
    for output_key in completed_outputs:
        output_content = str(results["outputs"][output_key])
        
        # Count paragraphs
        paragraphs = [p for p in output_content.split("\n\n") if p.strip()]
        
        # Count sentences (simple approximation)
        sentences = [s for s in output_content.replace("\n", " ").split(".") if s.strip()]
        
        print(f"  {output_key.capitalize()}: {len(paragraphs)} paragraphs, ~{len(sentences)} sentences")
    
    # Check for data transfer evidence
    print("\nData transfer analysis:")
    
    # Check if analysis references research
    if "research" in results["outputs"] and "analysis" in results["outputs"]:
        research_content = str(results["outputs"]["research"]).lower()
        analysis_content = str(results["outputs"]["analysis"]).lower()
        
        # Look for word overlap (simplified approach)
        research_words = set([w for w in research_content.split() if len(w) > 5][:20])  # Top 20 words
        overlap_count = sum(1 for word in research_words if word in analysis_content)
        overlap_percentage = (overlap_count / len(research_words)) * 100 if research_words else 0
        
        print(f"  Research → Analysis: {overlap_percentage:.1f}% keyword overlap")
    
    # Check if report references analysis
    if "analysis" in results["outputs"] and "report" in results["outputs"]:
        analysis_content = str(results["outputs"]["analysis"]).lower()
        report_content = str(results["outputs"]["report"]).lower()
        
        # Look for word overlap (simplified approach)
        analysis_words = set([w for w in analysis_content.split() if len(w) > 5][:20])  # Top 20 words
        overlap_count = sum(1 for word in analysis_words if word in report_content)
        overlap_percentage = (overlap_count / len(analysis_words)) * 100 if analysis_words else 0
        
        print(f"  Analysis → Report: {overlap_percentage:.1f}% keyword overlap")
    
    print("=" * 80)

# Analyze the workflow performance
analyze_workflow_performance(workflow_results)

## Conclusion

This notebook demonstrated Cerebrum's plugin handover capabilities, showing how different AI agent frameworks can collaborate in a unified workflow:

1. **Plugin System Flexibility**: We loaded and used three different agent frameworks through Cerebrum's plugin system.
2. **Seamless Data Handover**: Data was successfully passed between CrewAI, AutoGen, and SmolAgents plugins.
3. **Workflow Orchestration**: The system managed the execution flow and ensured outputs from each stage were properly formatted for the next.
4. **Visual Analysis**: We visualized the architecture and results to better understand the flow of information.

This demonstrates how Cerebrum can serve as a unifying framework for different AI agent technologies, allowing them to work together in complex workflows.